In [ ]:
import numpy as np
import pandas as pd
import rasterio as rio
import os
from tqdm import tqdm
from pathlib import Path

In [ ]:
ROOT = Path.cwd().parent
DATA = ROOT / 'data'
RESULTS = ROOT / 'results'

# 6 — Rarity-weighted richness (RWR) maps

Merges the former `6-make-species-richness-maps` and `7-make-rarity-richness-maps`. The pure species-richness (count) layers are dropped; this notebook produces only **rarity-weighted richness** — for each cell, the sum over species of (AOH in cell ÷ that species' total AOH), so geographically restricted species count more.

Per taxon and combined **vertebrates**, overall and by IUCN Red-List category (CR/EN/VU/NT), at 50 km — the inputs consumed by `6-biodiversity-risk-assessment` and the mining-area allocation (`3-mining-area-simulation/5-2`, `6`).

Output: `area-of-habitat-{res}/rarity-weighted-richness-{taxon|vertebrates}[-{category}]-{res}.tif`

In [ ]:
def make_rwr_for_taxon(taxon, category, res):
    """Rarity-weighted richness for one taxon: sum over species of (AOH in cell / species' total AOH)."""
    country_raster_ = rio.open(RESULTS / 'country-rasters' / f'country-raster-eck4-{res}.tif')
    country_raster = country_raster_.read(1)
    the_catalogue = pd.read_csv(RESULTS / f'area-of-habitat-{res}' / f'catalogue-{taxon}.csv')
    the_catalogue[['row_min', 'row_max', 'col_min', 'col_max']] = the_catalogue[['row_min', 'row_max', 'col_min', 'col_max']].astype(int)
    rwr_raster = np.zeros(country_raster_.shape, dtype=np.float32)
    if category == '':
        a_slice = the_catalogue[the_catalogue['have_a_AOH'] == True]
    else:
        a_slice = the_catalogue[(the_catalogue['have_a_AOH'] == True) & (the_catalogue['category'] == category)]
    for i in tqdm(a_slice.index):
        a_species = the_catalogue.loc[i, 'name']
        a_path = RESULTS / f'area-of-habitat-{res}' / taxon / f'{a_species}.tif'
        a_AOH = rio.open(a_path).read(1)
        row_min, row_max, col_min, col_max = the_catalogue.loc[i, ['row_min', 'row_max', 'col_min', 'col_max']]
        a_sum = np.sum(a_AOH)
        rwr_raster[row_min:row_max+1, col_min:col_max+1] += (a_AOH / a_sum)
    rwr_raster = np.where(country_raster > 0, rwr_raster, 0)
    suffix = '' if category == '' else f'-{category}'
    output_path = RESULTS / f'area-of-habitat-{res}' / f'rarity-weighted-richness-{taxon}{suffix}-{res}.tif'
    with rio.open(output_path, 'w', driver='GTiff',
        height=country_raster_.shape[0], width=country_raster_.shape[1], count=1,
        dtype=rwr_raster.dtype, crs=country_raster_.crs,
        transform=country_raster_.transform, compress='lzw') as dst:
        dst.write(rwr_raster, 1)
    return rwr_raster

In [ ]:
def make_rwr_for_vertebrates(taxon_arrays, category, res):
    """Combine per-taxon RWR rasters into a vertebrates RWR layer."""
    country_raster_ = rio.open(RESULTS / 'country-rasters' / f'country-raster-eck4-{res}.tif')
    rwr = np.nansum(taxon_arrays, axis=0).astype(np.float32)
    suffix = '' if category == '' else f'-{category}'
    output_path = RESULTS / f'area-of-habitat-{res}' / f'rarity-weighted-richness-vertebrates{suffix}-{res}.tif'
    with rio.open(output_path, 'w', driver='GTiff',
        height=country_raster_.shape[0], width=country_raster_.shape[1], count=1,
        dtype=rwr.dtype, crs=country_raster_.crs, transform=country_raster_.transform, compress='lzw') as dst:
        dst.write(rwr, 1)
    return rwr

## Run — per-taxon + vertebrates RWR (overall + by IUCN category), 50 km

In [ ]:
TAXA = ['mammals', 'birds', 'amphibians', 'reptiles']
CATEGORIES = ['', 'CR', 'EN', 'VU', 'NT']
RES = '50km'

rwr = {}
for taxon in TAXA:
    for cat in CATEGORIES:
        rwr[(taxon, cat)] = make_rwr_for_taxon(taxon, cat, RES)

for cat in CATEGORIES:
    make_rwr_for_vertebrates([rwr[(t, cat)] for t in TAXA], cat, RES)